# RAG local para sessão de perguntas e resposta sobre o cardápio de hoje

- Gera embedding a partir de informações estruturadas da imagem do cardápio, usando modelo disponibilizado no Hugging Face

- Cria banco vetorial e realiza buscas semânticas
  
- Realiza sessão de perguntas e respostassobre o cardápio

## Usa RAG para sessão de perguntas & respostas

In [ ]:
# image input 
img_path = 'input_imgs/cardapio-hoje.png'

# output path
csv_path = 'outputs_json/cardapio-hoje-infos-extracted.csv'

In [ ]:
import os
from rich import print
from dotenv import load_dotenv
import re
import pandas as pd

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq

In [5]:
df_menu = pd.read_csv(csv_path)
df_menu.head()

,item_name,price_P,price_M,price_G,description,price
0,Tradicional,45.0,45.0,55.0,"Queijo, presunto, tomate, azeitona preta",NaN
1,SMASH BURGER,NaN,NaN,NaN,"Hamburger Bevin Pao Especial, Sabskcha, Guaje,...",NaN
2,Calabresa,NaN,NaN,NaN,"Queija, Mahe Eszecial e Maths Especial",NaN
3,TOP BURGER,NaN,NaN,NaN,"Hamburger Bevino, Que ja Pao Especial, Salsich...",NaN
4,Portuguesa,45.0,45.0,55.0,"Queijo presunto, tomate, azeitona preta",NaN


In [6]:
docs = []
for _, row in df_menu.iterrows():

    text = ' | '.join([f'{col} = {row[col]}' for col in df_menu.columns])

    docs.append(
        Document(
            page_content=text,
            metadata=row.to_dict()
        )
    )

In [7]:
docs

[Document(metadata={'item_name': 'Tradicional', 'price_P': 45.0, 'price_M': 45.0, 'price_G': 55.0, 'description': 'Queijo, presunto, tomate, azeitona preta', 'price': nan}, page_content='item_name = Tradicional | price_P = 45.0 | price_M = 45.0 | price_G = 55.0 | description = Queijo, presunto, tomate, azeitona preta | price = nan'),
 Document(metadata={'item_name': 'SMASH BURGER', 'price_P': nan, 'price_M': nan, 'price_G': nan, 'description': 'Hamburger Bevin Pao Especial, Sabskcha, Guaje, Batata Falke', 'price': nan}, page_content='item_name = SMASH BURGER | price_P = nan | price_M = nan | price_G = nan | description = Hamburger Bevin Pao Especial, Sabskcha, Guaje, Batata Falke | price = nan'),
 Document(metadata={'item_name': 'Calabresa', 'price_P': nan, 'price_M': nan, 'price_G': nan, 'description': 'Queija, Mahe Eszecial e Maths Especial', 'price': nan}, page_content='item_name = Calabresa | price_P = nan | price_M = nan | price_G = nan | description = Queija, Mahe Eszecial e Math

In [8]:
embed_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-base-en-v1.5'
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
emb_test = embed_model.embed_query('What is Retrieval Augmented Generation?')
print(emb_test[:10])

[
    -0.024018358439207077,
    -0.026337644085288048,
    0.0034917350858449936,
    0.05912458151578903,
    0.031969547271728516,
    -0.0069933668710291386,
    -0.005698849447071552,
    0.001487469649873674,
    -0.03619713336229324,
    -0.0025471162516623735
]

In [10]:
vectorstore = FAISS.from_documents(
    docs,
    embed_model
)

In [11]:
question = 'Qual pizza tem no cardápio?'

In [12]:
docs_retrieved = vectorstore.similarity_search(question, k=5)
docs_retrieved

[Document(id='431aff2d-0557-495f-8648-51903d408f91', metadata={'item_name': 'Pizza a moda da casa', 'price_P': 45.0, 'price_M': 45.0, 'price_G': 55.0, 'description': 'Queijo, presunto, tomate, azeitona preta', 'price': nan}, page_content='item_name = Pizza a moda da casa | price_P = 45.0 | price_M = 45.0 | price_G = 55.0 | description = Queijo, presunto, tomate, azeitona preta | price = nan'),
 Document(id='cfe36a94-748f-4d8f-9d74-a17031db07ad', metadata={'item_name': 'CHICKEN BURGER', 'price_P': 45.0, 'price_M': 45.0, 'price_G': 55.0, 'description': 'Frango com requeijao, Queijo, presunto, tomate, azeitona preta', 'price': nan}, page_content='item_name = CHICKEN BURGER | price_P = 45.0 | price_M = 45.0 | price_G = 55.0 | description = Frango com requeijao, Queijo, presunto, tomate, azeitona preta | price = nan'),
 Document(id='6b60543e-dea4-4560-accb-da256c35f9e1', metadata={'item_name': 'TOP BURGER', 'price_P': nan, 'price_M': nan, 'price_G': nan, 'description': 'Hamburger Bevino, Qu

In [ ]:
llm_model = ChatGroq(
        model='qwen/qwen3-32b',
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
        verbose=False
    )

In [15]:
context = '\n'.join([doc.page_content for doc in docs_retrieved])
prompt = f'Use o seguinte contexto para responder à pergunta:\n{context}\nPergunta: {question}'

In [16]:
print(prompt)

Use o seguinte contexto para responder à pergunta:
item_name = Pizza a moda da casa | price_P = 45.0 | price_M = 45.0 | price_G = 55.0 | description = Queijo, 
presunto, tomate, azeitona preta | price = nan
item_name = CHICKEN BURGER | price_P = 45.0 | price_M = 45.0 | price_G = 55.0 | description = Frango com requeijao,
Queijo, presunto, tomate, azeitona preta | price = nan
item_name = TOP BURGER | price_P = nan | price_M = nan | price_G = nan | description = Hamburger Bevino, Que ja Pao
Especial, Salsicha, Qurija | price = nan
item_name = VEGANO BURGER | price_P = nan | price_M = nan | price_G = nan | description = Pao Especial, Hambdrger 
Vegetal Quai | price = nan
item_name = Calabresa | price_P = nan | price_M = nan | price_G = nan | description = Queija, Mahe Eszecial e Maths
Especial | price = nan
Pergunta: Qual pizza tem no cardápio?

In [17]:
system_prompt = 'Você é um assistente que responde pergunta sobre cardápio de restaurante.'
messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content=prompt)
]

response = llm_model.invoke(messages)

In [18]:
print(response.content)

<think>
Okay, let's see. The user is asking which pizza is on the menu. Looking at the provided context, there are several 
items listed. The first one is "Pizza a moda da casa" with prices for P, M, G, but the main price is NaN. The 
description mentions cheese, ham, tomato, black olive. Then there are burgers like Chicken Burger, Top Burger, 
Vegan Burger, and Calabresa. The Calabresa has a description with cheese, special dough, and special sauce. Wait, 
the user is asking about pizza, so I need to check if any of these items are pizzas.

The first item is clearly a pizza: "Pizza a moda da casa". The others are burgers. The Calabresa might be a type of
pizza, but the description says "Queija, Mahe Eszecial e Maths Especial" which seems like cheese, special dough, 
and special sauce. But the name "Calabresa" is a type of pizza in some regions. However, the user specifically 
asked for pizza, so the first item is the one explicitly named as pizza. The others are burgers. So the answer 
should be "Pizza a moda da casa".
</think>

A pizza disponível no cardápio é a **Pizza a moda da casa**, com os seguintes ingredientes: queijo, presunto, 
tomate e azeitona preta. As opções de tamanho e preço são:  
- Pequena (P): R$45,00  
- Média (M): R$45,00  
- Grande (G): R$55,00  

Os demais itens listados são hambúrgueres (Chicken Burger, Top Burger, Vegano Burger) e uma pizza Calabresa (com 
descrição incompleta), mas a única pizza claramente identificada é a "Pizza a moda da casa".

## Usa RAG para sessão de perguntas & respostas

In [19]:
def menu_agent(question: str) -> str:
    docs_retrieved = vectorstore.similarity_search(question, k=5)

    context = '\n'.join([doc.page_content for doc in docs_retrieved])
    prompt = f'Use o seguinte contexto para responder à pergunta:\n{context}\nPergunta: {question}'
    
    system_prompt = 'Você é um assistente que responde pergunta sobre cardápio de restaurante.'
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=prompt)
    ]

    res = llm_model.invoke(messages).content

    res_clean = re.sub(r'<think>.*?</think>', '', res, flags=re.DOTALL).strip()

    return res_clean

In [20]:
res = menu_agent(question=question)

In [ ]:

print(res)

A pizza disponível no cardápio é a **Pizza a moda da casa**, com os seguintes ingredientes: queijo, presunto, 
tomate e azeitona preta. Os preços são:  
- **P (Pequena): R$ 45,00**  
- **M (Média): R$ 45,00**  
- **G (Grande): R$ 55,00**  

As demais opções listadas são hambúrgueres (Chicken Burger, Top Burger, Vegano Burger) e uma entrada ou pizza não 
especificada chamada "Calabresa", mas não estão classificadas como pizzas no cardápio.

In [22]:
####################
# Alguma perguntas #
####################

# Hoje temos pizza?
# Tem hamburguer?
# Tem algum item com carne?
# Tem algum item sem carne?
# Qual item mais barato?

In [ ]:

print('===== Entre com suas questões (pressione Enter para sair) =====')
while 1 > 0:
    q = input('Humano: ')
    if not len(q):
        break
    else:
        print(f'🙋‍♀️ Pergunta: {q}')
        ans = menu_agent(question=q)
        print(f'🤖 Resposta:\n{ans}')
        print(50*'=')

===== Entre com suas questões (pressione Enter para sair) =====

🙋‍♀️ Pergunta: Boa noite. Hoje temos pizza?

🤖 Resposta:
Boa noite! Sim, hoje temos duas opções de pizza:  

1. **Pizza a moda da casa**  
   - **Ingredientes:** Queijo, presunto, tomate, azeitona preta  
   - **Preços:** R$ 45,00 (P/M) e R$ 55,00 (G)  

2. **Pizza Portuguesa**  
   - **Ingredientes:** Queijo, presunto, tomate, azeitona preta  
   - **Preços:** R$ 45,00 (P/M) e R$ 55,00 (G)  

Ambas têm os mesmos ingredientes, mas podem variar conforme a preparação da casa. Também temos opções de 
hambúrgueres, como o **Chicken Burger** e o **Top Burger**, caso prefira. Deseja mais detalhes sobre alguma opção? 
😊

==================================================

🙋‍♀️ Pergunta: Tem hamburguer?

...

🤖 Resposta:
Sim, há um item sem carne no cardápio: **VEGANO BURGER**. Ele é descrito como "Pão Especial, Hambúrguer Vegetal 
Quai", indicando que utiliza um hambúrguer vegetal (não carnoso) como ingrediente principal. Os demais itens (TOP 
BURGER, CHICKEN BURGER, Calabresa e Camardão) contêm carne ou derivados animais.

==================================================

🙋‍♀️ Pergunta: Qual item mais barato?

🤖 Resposta:
O item mais barato está disponível em três opções, todas com o preço de **R$45,00** nas versões P e M:  
1. **CHICKEN BURGER**  
2. **Pizza a moda da casa**  
3. **Portuguesa**  

Esses itens compartilham o mesmo preço mínimo.

==================================================

### Cardápio do contexto

In [24]:
df_menu

,item_name,price_P,price_M,price_G,description,price
0,Tradicional,45.0,45.0,55.0,"Queijo, presunto, tomate, azeitona preta",NaN
1,SMASH BURGER,NaN,NaN,NaN,"Hamburger Bevin Pao Especial, Sabskcha, Guaje,...",NaN
2,Calabresa,NaN,NaN,NaN,"Queija, Mahe Eszecial e Maths Especial",NaN
3,TOP BURGER,NaN,NaN,NaN,"Hamburger Bevino, Que ja Pao Especial, Salsich...",NaN
4,Portuguesa,45.0,45.0,55.0,"Queijo presunto, tomate, azeitona preta",NaN
5,CHICKEN BURGER,45.0,45.0,55.0,"Frango com requeijao, Queijo, presunto, tomate...",NaN
6,VEGANO BURGER,NaN,NaN,NaN,"Pao Especial, Hambdrger Vegetal Quai",NaN
7,Camardao,NaN,NaN,NaN,"Queijo, presunto, tomate, azeitona preta",NaN
8,MONSTRO BURGER,NaN,NaN,NaN,2 Hamburger de Picatha,NaN
9,Pizza a moda da casa,45.0,45.0,55.0,"Queijo, presunto, tomate, azeitona preta",NaN


<img src='input_imgs/cardapio-hoje.png'>